In [ ]:
import numpy as np
import pandas as pd
import re
from datetime import datetime
       

 # The star of the show
from google_play_scraper import app, reviews, Sort      

import sys,os 
sys.path.insert(0, os.path.abspath('..'))

---
## 1. Scraping Reviews

Now let's collect the actual reviews. The `reviews()` function lets us specify:
- **How many** reviews to fetch (`count`)
- **Which order** to sort them (`Sort.NEWEST` or `Sort.MOST_RELEVANT`)
- **Filter by rating** (we want all ratings, so `None`)


In [ ]:
from src.data_scrapping import scrape_fintech_reviews, APPS

df_raw = scrape_fintech_reviews(APPS, count=500)
df_raw = pd.DataFrame(df_raw)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

---
## 2. EDA and Data Quality Audit

> **"Garbage In, Garbage Out (GIGO)"** — If our input data is flawed, every insight we build on top of it is also flawed.

Let's audit our three common data quality problems:
1. Missing values
2. Duplicate reviews  
3. Inconsistent date formats

In [ ]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(3).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

In [ ]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")


In [ ]:
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")
print(df_raw.head(3))

---
## 3. Cleaning Strategy

Now we fix each problem, one at a time.  
We work on a **copy** so we can always compare back to the raw data.

### Strategy Overview

| Problem | Strategy | Tool |
|---------|----------|------|
| Missing critical data | Drop the row | `df.dropna()` |
| Duplicate reviews | Keep first occurrence | `df.drop_duplicates()` |
| Inconsistent dates | Parse and reformat | `pd.to_datetime()` |
| Messy text | Strip whitespace / remove junk | `str.strip()`, `re.sub()` |

In [ ]:
# Work on a copy so raw data stays untouched
df = df_raw.copy()
print(df_raw.head(3))
print(f"Starting with: {len(df)} reviews")

### 3.1 Handle Missing Values

The **review text** and **rating** are critical — without them, the row is useless for analysis.  
We drop any row missing these. Other fields (like `review_id`) are less critical.

In [ ]:
missing_review = df['review'].isna().sum()
missing_rating = df['rating'].isna().sum()

before_dropna = len(df)

df = df.dropna(subset=['review', 'rating'])


after_dropna = len(df)
dropped = before_dropna - after_dropna
print(f"\n2. Missing values handled:")
print(f"   - Rows with missing review text: {missing_review}")
print(f"   - Total rows dropped: {dropped}")
print(f"   Shape after dropna: {after_dropna}")



### 3.2 Remove duplicate

In [ ]:
before_dedup = len(df)
df = df.drop_duplicates(
    subset=['review', 'rating', 'date', 'bank'], 
    keep='first'
).copy()
after_dedup = len(df)
print(f"\n1. Duplicates removed: {before_dedup - after_dedup} rows")
print(f"   Shape before dedup: {before_dedup}")
print(f"   Shape after dedup: {after_dedup}")

### 3.3 Normalize dates
Dates need to be in a consistent `YYYY-MM-DD` format for:
- Time-series analysis (sentiment over time)
- Sorting
- Database storage

The `date` field from the scraper is a Python `datetime` object. We convert it to a clean date string.


In [ ]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
print(f"\n3. Dates normalized to YYYY-MM-DD format")
print(df['date'].head(3))

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

### 3.4 Cleaning Review tests
- remove whitespaces and newlines
- empty strings and spaces

In [ ]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")